In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymannkendall as mk
from matplotlib.lines import Line2D
from matplotlib import font_manager

# === Input file path ===
csv_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Forest_Area_Change_historical_Disturbance_Attribution_1988_2021_pa&ownership_reprojected.csv"

# === Disturbance categories and colors (merged) ===
disturbance_labels = [
    "Logging", "Construction", "Stress", "Natural Hazard",
    "Water Dynamic", "Fire", "Agriculture Activity", "Others"
]

# label → color (keeps your original hues; “Others” is gray)
disturbance_color_map = {
    "Logging": "#66c2a5",
    "Construction": "purple",
    "Stress": "#f781bf",
    "Natural Hazard": "#a65628",
    "Water Dynamic": "#1f78b4",
    "Fire": "red",
    "Agriculture Activity": "#ffd92f",
    "Others": "#b3b3b3"
}


# === US state to region and division mapping ===
state_to_region_division = {
    'Connecticut': ('Northeast', 'New England'), 'Maine': ('Northeast', 'New England'),
    'Massachusetts': ('Northeast', 'New England'), 'New Hampshire': ('Northeast', 'New England'),
    'Rhode Island': ('Northeast', 'New England'), 'Vermont': ('Northeast', 'New England'),
    'New Jersey': ('Northeast', 'Middle Atlantic'), 'New York': ('Northeast', 'Middle Atlantic'),
    'Pennsylvania': ('Northeast', 'Middle Atlantic'),
    'Illinois': ('Midwest', 'East North Central'), 'Indiana': ('Midwest', 'East North Central'),
    'Michigan': ('Midwest', 'East North Central'), 'Ohio': ('Midwest', 'East North Central'),
    'Wisconsin': ('Midwest', 'East North Central'),
    'Iowa': ('Midwest', 'West North Central'), 'Kansas': ('Midwest', 'West North Central'),
    'Minnesota': ('Midwest', 'West North Central'), 'Missouri': ('Midwest', 'West North Central'),
    'Nebraska': ('Midwest', 'West North Central'), 'North Dakota': ('Midwest', 'West North Central'),
    'South Dakota': ('Midwest', 'West North Central'),
    'Delaware': ('South', 'South Atlantic'), 'District of Columbia': ('South', 'South Atlantic'),
    'Florida': ('South', 'South Atlantic'), 'Georgia': ('South', 'South Atlantic'),
    'Maryland': ('South', 'South Atlantic'), 'North Carolina': ('South', 'South Atlantic'),
    'South Carolina': ('South', 'South Atlantic'), 'Virginia': ('South', 'South Atlantic'),
    'West Virginia': ('South', 'South Atlantic'),
    'Alabama': ('South', 'East South Central'), 'Kentucky': ('South', 'East South Central'),
    'Mississippi': ('South', 'East South Central'), 'Tennessee': ('South', 'East South Central'),
    'Arkansas': ('South', 'West South Central'), 'Louisiana': ('South', 'West South Central'),
    'Oklahoma': ('South', 'West South Central'), 'Texas': ('South', 'West South Central'),
    'Arizona': ('West', 'Mountain'), 'Colorado': ('West', 'Mountain'), 'Idaho': ('West', 'Mountain'),
    'Montana': ('West', 'Mountain'), 'Nevada': ('West', 'Mountain'), 'New Mexico': ('West', 'Mountain'),
    'Utah': ('West', 'Mountain'), 'Wyoming': ('West', 'Mountain'),
    'Alaska': ('West', 'Pacific'), 'California': ('West', 'Pacific'),
    'Hawaii': ('West', 'Pacific'), 'Oregon': ('West', 'Pacific'), 'Washington': ('West', 'Pacific')
}

# === Load data ===
df = pd.read_csv(csv_path)

# Identify the column representing state or ecoregion
state_col = next((col for col in df.columns if col.lower() in ['state', 'ecoregion', 'name']), None)
if state_col is None:
    raise ValueError("Cannot find a column representing State or Ecoregion!")

# Map each row to a region
df["Region"] = df[state_col].map(lambda s: state_to_region_division.get(s, (np.nan,))[0])
df = df[~df["Region"].isna()]  # Drop rows with unmapped regions
#df = df[df["GapYears"] == 0]    # Filter for zero gap years only

# Split ForestChangeType into from_class and to_class
df["from_class"] = df["ForestChangeType"].astype(str).str[0]
df["to_class"] = df["ForestChangeType"].astype(str).str[1]

# === Calculate fragmentation numerator: internal forest -> edge forest ===
fragment_mask = (df["from_class"] == "5") & (df["to_class"].isin(['1', '2', '3', '4']))
frag_df = df[fragment_mask]
frag_trend = (
    frag_df.groupby(["Year_From", "DisturbanceCategory", "Region"])["PixelCount"]
    .sum().reset_index().rename(columns={"PixelCount": "EFCPixels"})
)

# === Calculate denominator: total forest loss (1/2/3/4/5 -> 0) ===
forest_loss_mask = (df["from_class"].isin(['1', '2', '3', '4', '5'])) & (df["to_class"] == '0')
loss_df = df[forest_loss_mask]
loss_trend = (
    loss_df.groupby(["Year_From", "DisturbanceCategory", "Region"])["PixelCount"]
    .sum().reset_index().rename(columns={"PixelCount": "TotalLossPixels"})
)

# === Calculate fragmentation ratio ===
trend_norm = pd.merge(frag_trend, loss_trend, on=["Year_From", "DisturbanceCategory", "Region"], how="outer").fillna(0)
# --- Normalize labels and MERGE "No Disturbance Detected" + "Other" → "Others" ---
trend_norm["DisturbanceCategory"] = trend_norm["DisturbanceCategory"].replace({
    "No Disturbance": "No Disturbance Detected",  # unify raw name
    "Forest Management": "Logging",               # your earlier mapping
    "Other": "Others",
    "No Disturbance Detected": "Others"
})

# Collapse after relabeling (sum numerators/denominators), then recompute EFCR
trend_norm = (
    trend_norm
    .groupby(["Year_From", "Region", "DisturbanceCategory"], as_index=False)[["EFCPixels", "TotalLossPixels"]]
    .sum()
)
trend_norm["EFCR"] = trend_norm["EFCPixels"] / trend_norm["TotalLossPixels"]
trend_norm.replace([np.inf, -np.inf], np.nan, inplace=True)


# Get region list and assign colors
regions = trend_norm["Region"].unique()
region_colors = dict(zip(regions, ['#1f78b4', '#33a02c', '#ff7f00', '#e31a1c']))

# === Mann-Kendall trend test and summary statistics ===
# === Mann-Kendall (no zero-fill) + UNWEIGHTED mean EFCR across years ===
summary_data = []
years_min = int(trend_norm["Year_From"].min())
years_max = int(trend_norm["Year_From"].max())

for disturbance in disturbance_labels:
    sub_df = trend_norm[trend_norm["DisturbanceCategory"] == disturbance]
    for region in regions:
        region_sub = sub_df[sub_df["Region"] == region]
        if region_sub.empty:
            continue

        # 1) Yearly totals (no reindex/zero-fill)
        g = (region_sub
             .groupby("Year_From", as_index=True)[["EFCPixels", "TotalLossPixels"]]
             .sum())

        # 2) Per-year EFCR; undefined when loss==0
        g["EFCR"] = np.where(g["TotalLossPixels"] > 0,
                             g["EFCPixels"] / g["TotalLossPixels"],
                             np.nan)

        # 3) UNWEIGHTED mean across years with data (each year = 1 vote)
        mean_efcr_unweighted = g["EFCR"].mean(skipna=True)

        # 4) MK trend on years with data only (no zero-fill)
        efcr_series = g["EFCR"].dropna().values
        if efcr_series.size >= 3:
            mk_result = mk.original_test(efcr_series)
            sen_slope = mk_result.slope      # units: EFCR per year
            p_value   = mk_result.p
        else:
            sen_slope = np.nan
            p_value   = np.nan

        summary_data.append({
            "Disturbance": disturbance,
            "Region": region,
            "Slope": sen_slope,
            "MeanEFCR": mean_efcr_unweighted,
            "p-value": p_value
        })

summary_df = pd.DataFrame(summary_data)

# === Assign unique markers for each disturbance category ===
disturbance_markers = ['o', 's', '^', 'D', 'v', 'P', '*', 'X', '<', '>']
marker_map = {dist: disturbance_markers[i % len(disturbance_markers)] for i, dist in enumerate(disturbance_labels)}

# === Calculate mean EFCR per disturbance category (across all regions) ===
efcr_by_disturbance = (
    summary_df.groupby("Disturbance")["MeanEFCR"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
# === National & Northeast EFCR trend summary ===
import pymannkendall as mk
import numpy as np
import pandas as pd

def calc_region_efcr_trend(trend_norm, region=None):
    if region:
        df = trend_norm[trend_norm["Region"] == region]
    else:
        df = trend_norm.copy()

    g = (
        df.groupby("Year_From")[["EFCPixels", "TotalLossPixels"]]
          .sum()
          .sort_index()
    )
    g["EFCR"] = np.where(g["TotalLossPixels"] > 0,
                         g["EFCPixels"] / g["TotalLossPixels"],
                         np.nan)

    efcr_series = g["EFCR"].dropna().values

    mk_res = mk.original_test(efcr_series)
    return {
        "Region": "National" if region is None else region,
        "Years": f"{int(g.index.min())}-{int(g.index.max())}",
        "Mean_EFCR": np.nanmean(efcr_series),
        "Sen_Slope_per_year": mk_res.slope,
        "p_value": mk_res.p,
        "Tau": mk_res.Tau
    }

national_trend  = calc_region_efcr_trend(trend_norm, region=None)
northeast_trend = calc_region_efcr_trend(trend_norm, region="Northeast")

trend_summary_df = pd.DataFrame([national_trend, northeast_trend])
print("\n=== EFCR Trend Summary (National & Northeast) ===")
print(trend_summary_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib import font_manager

# === Create 3-column layout: bar, scatter, legend ===
fig = plt.figure(figsize=(18, 6))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 0.9, 0.3])

sns.set(style="whitegrid")

# ===== Subplot (a): Bar chart =====
ax1 = fig.add_subplot(gs[0])
bars = ax1.bar(
    efcr_by_disturbance.index,
    efcr_by_disturbance.values,
    color=[disturbance_color_map[d] for d in efcr_by_disturbance.index]
)
ax1.set_xticklabels(efcr_by_disturbance.index, rotation=-30, ha="center")
ax1.set_ylabel("Mean EFCR")
ax1.set_title("(a) Mean EFCR by Disturbance Type", fontsize=12, fontweight='bold', loc='left')

# ===== Subplot (b): Scatter plot =====
ax2 = fig.add_subplot(gs[1])

region_marker_map = {
    region: marker for region, marker in zip(region_colors.keys(), ['o', 's', '^', 'D', 'P', 'X'])
}

for _, row in summary_df.iterrows():
    is_significant = row["p-value"] <= 0.05
    ax2.scatter(
        row["MeanEFCR"],
        row["Slope"],
        color=disturbance_color_map[row["Disturbance"]] if is_significant else 'white',
        edgecolor=disturbance_color_map[row["Disturbance"]],
        marker=region_marker_map[row["Region"]],
        linewidth=1.2,
        s=120
    )


ax2.set_title("(b) EFCRs and Their Trend by Disturbance Type and Region", fontsize=12, fontweight='bold', loc='left')
ax2.set_ylabel("Sen's Slope of EFCR")
ax2.set_xlabel("Mean EFCR")

# ===== Subplot (c): Legend block =====
ax3 = fig.add_subplot(gs[2])
ax3.axis("off")

# -- Region (Shape)
region_handles = [
    Line2D([0], [0], marker=region_marker_map[region], color='black', label=region,
           linestyle='None', markerfacecolor='black', markersize=10)
    for region in region_colors
]

# -- Disturbance (Color)
disturbance_handles = [
    Line2D([0], [0], marker='o', color=disturbance_color_map[d], label=d,
           linestyle='None', markerfacecolor=disturbance_color_map[d],
           markeredgecolor='black', markersize=10)
    for d in disturbance_labels
]


# -- Significance (Fill)
significance_handles = [
    Line2D([0], [0], marker='o', color='gray', linestyle='None',
           label='Significant (p ≤ 0.05)', markerfacecolor='gray', markeredgecolor='gray', markersize=10),
    Line2D([0], [0], marker='o', color='gray', linestyle='None',
           label='Not Significant', markerfacecolor='white', markeredgecolor='gray', markersize=10)
]

# Add 3 legends stacked vertically
legend1 = ax3.legend(
    handles=region_handles,
    title="Region",
    loc='upper left',
    bbox_to_anchor=(-0.28, 1.0),
    frameon=False,
    title_fontproperties=font_manager.FontProperties(weight='bold')
)
ax3.add_artist(legend1)

# Disturbance Legend – middle
legend2 = ax3.legend(
    handles=disturbance_handles,
    title="Disturbance",
    loc='upper left',
    bbox_to_anchor=(-0.28, 0.65),
    frameon=False,
    title_fontproperties=font_manager.FontProperties(weight='bold')
)
ax3.add_artist(legend2)

# Trend Significance Legend – bottom
legend3 = ax3.legend(
    handles=significance_handles,
    title="Trend Significance",
    loc='upper left',
    bbox_to_anchor=(-0.28, 0.15),
    frameon=False,
    title_fontproperties=font_manager.FontProperties(weight='bold')
)

legend1._legend_box.align = "left"
legend2._legend_box.align = "left"
legend3._legend_box.align = "left"

# === Final layout and save ===
plt.tight_layout()
plt.savefig(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Figure_3.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
summary_df

In [ ]:
# === EFCR by year for Logging in the Northeast ===

# Get full year range available in trend_norm
years = np.arange(trend_norm["Year_From"].min(), trend_norm["Year_From"].max() + 1)

# Subset for Logging in Northeast
ne_log = trend_norm[
    (trend_norm["DisturbanceCategory"] == "Logging") &
    (trend_norm["Region"] == "Northeast")
].copy()

# Rebuild year-complete table (years with no loss = 0)
ne_log = ne_log.groupby("Year_From", as_index=False)[["EFCPixels","TotalLossPixels"]].sum()
ne_log = ne_log.set_index("Year_From").reindex(years).fillna(0)
ne_log["EFCR"] = np.where(ne_log["TotalLossPixels"] > 0,
                          ne_log["EFCPixels"] / ne_log["TotalLossPixels"], np.nan)

# Weighted EFCR (definition: sum EFCPixels / sum TotalLossPixels)
def weighted_efcr(df):
    num = df["EFCPixels"].sum()
    den = df["TotalLossPixels"].sum()
    return num / den if den > 0 else np.nan

# Weighted trend (weighted OLS slope)
def weighted_ols_trend(df):
    g = df.dropna(subset=["EFCR"]).copy()
    x = g.index.to_numpy().astype(float)
    y = g["EFCR"].to_numpy().astype(float)
    w = g["TotalLossPixels"].to_numpy().astype(float)
    if w.sum() == 0:
        return np.nan
    xbar = (w * x).sum() / w.sum()
    ybar = (w * y).sum() / w.sum()
    slope = ((w * (x - xbar) * (y - ybar)).sum()) / ((w * (x - xbar)**2).sum())
    return slope

# First 5 years (chronologically)
first_5_years = ne_log.iloc[:5]
last_5_years = ne_log.iloc[-5:]

mean_first5 = weighted_efcr(first_5_years)
mean_last5  = weighted_efcr(last_5_years)
mean_total  = weighted_efcr(ne_log)

trend_slope = weighted_ols_trend(ne_log)

print("Logging · Northeast")
print(f"Weighted EFCR (first 5 yrs): {mean_first5:.3f}")
print(f"Weighted EFCR (last 5 yrs):  {mean_last5:.3f}")
print(f"Weighted EFCR (total):       {mean_total:.3f}")
print(f"Weighted trend (per year):   {trend_slope:.3f}")


In [ ]:
# Build per-year EFCR without reindex/zero-fill
g = (trend_norm[(trend_norm["DisturbanceCategory"]=="Logging") &
                (trend_norm["Region"]=="Northeast")]
     .groupby("Year_From", as_index=True)[["EFCPixels","TotalLossPixels"]]
     .sum())
g["EFCR"] = np.where(g["TotalLossPixels"]>0,
                     g["EFCPixels"]/g["TotalLossPixels"], np.nan)

# Use years that actually have EFCR
g_valid = g.dropna(subset=["EFCR"]).copy()

# First/last 5 years by actual data
g_first5 = g_valid.head(5)
g_last5  = g_valid.tail(5)

mean_first5_unw = g_first5["EFCR"].mean()
mean_last5_unw  = g_last5["EFCR"].mean()
mean_total_unw  = g_valid["EFCR"].mean()

# Unweighted OLS slope (EFCR units per year)
x = g_valid.index.values.astype(float)
y = g_valid["EFCR"].values.astype(float)
ols_slope_unw = np.polyfit(x, y, 1)[0]

print("Logging · Northeast (UNWEIGHTED)")
print(f"Mean EFCR (first 5 yrs): {mean_first5_unw:.3f}")
print(f"Mean EFCR (last 5 yrs):  {mean_last5_unw:.3f}")
print(f"Mean EFCR (total):       {mean_total_unw:.3f}")
print(f"OLS trend (per year):    {ols_slope_unw:.3f}")


In [ ]:
# === Helpers: weighted mean + CI, Sen's slope + CI =============================================

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w >= 0)
    x, w = x[mask], w[mask]
    if w.sum() == 0:
        return np.nan
    return np.sum(w * x) / np.sum(w)

def bootstrap_weighted_mean_ci(x, w, B=2000, alpha=0.05, random_state=42):
    """
    Nonparametric bootstrap across years. Each bootstrap draw resamples years with replacement,
    then recomputes the weighted mean using the resampled (x_i, w_i).
    """
    rng = np.random.default_rng(random_state)
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w)
    x, w = x[mask], w[mask]
    n = len(x)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    boot = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        x_b = x[idx]; w_b = w[idx]
        wm = weighted_mean(x_b, w_b)
        boot[b] = wm
    lo = np.nanpercentile(boot, 100*alpha/2)
    hi = np.nanpercentile(boot, 100*(1 - alpha/2))
    return (weighted_mean(x, w), lo, hi)

def sen_slope_and_ci(years, y, alpha=0.05):
    """
    Sen's slope via pymannkendall + 95% CI using Hirsch et al. method.
    Requires at least 3 unique years with finite EFCR.
    """
    years = np.asarray(years, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(years) & np.isfinite(y)
    years, y = years[m], y[m]
    # sort by year to be safe
    order = np.argsort(years)
    years, y = years[order], y[order]
    # drop duplicate years by averaging EFCR within the same year (rare edge case)
    _, idx_unique = np.unique(years, return_index=True)
    if len(idx_unique) < len(years):
        df_tmp = pd.DataFrame({"year": years, "y": y}).groupby("year")["y"].mean().reset_index()
        years = df_tmp["year"].values
        y = df_tmp["y"].values

    n = len(y)
    if n < 3:
        return dict(slope=np.nan, p=np.nan, lo=np.nan, hi=np.nan)

    # Mann–Kendall original test
    mk_res = mk.original_test(y)  # years are equally spaced; OK to use y only
    slope = mk_res.slope
    pval = mk_res.p
    var_s = mk_res.var_s

    # Build all pairwise slopes
    slopes = []
    for j in range(1, n):
        dy = y[j] - y[:j]
        dx = years[j] - years[:j]
        s = dy / dx
        slopes.append(s)
    slopes = np.sort(np.concatenate(slopes))
    Np = len(slopes)  # should be n*(n-1)/2

    # 95% CI using normal approximation to S
    if var_s is None or var_s <= 0 or Np == 0:
        return dict(slope=slope, p=pval, lo=np.nan, hi=np.nan)

    z = 1.959963984540054  # ~ N(0,1) 97.5% quantile
    C_alpha = z * np.sqrt(var_s)

    # Rank positions for CI bounds (Helsel & Hirsch formulation)
    M1 = int(np.ceil((Np - C_alpha) / 2.0))
    M2 = int(np.ceil((Np + C_alpha) / 2.0))
    # Convert to 0-based safe indices
    L = max(0, min(Np - 1, M1 - 1))
    U = max(0, min(Np - 1, M2 - 1))
    lo = slopes[L]
    hi = slopes[U]
    if lo > hi:  # just in case of degenerate indexing
        lo, hi = hi, lo

    return dict(slope=slope, p=pval, lo=lo, hi=hi)

# === Build national & regional EFCR time series ================================================

# National time series (aggregate across all regions & disturbances)
national_yearly = (
    trend_norm
    .groupby("Year_From")[["EFCPixels", "TotalLossPixels"]]
    .sum()
    .reset_index()
)
national_yearly["EFCR"] = national_yearly["EFCPixels"] / national_yearly["TotalLossPixels"]
national_yearly.replace([np.inf, -np.inf], np.nan, inplace=True)
national_yearly = national_yearly.dropna(subset=["EFCR", "TotalLossPixels"])

# Per-region time series (aggregate across disturbances)
region_yearly = (
    trend_norm
    .groupby(["Region", "Year_From"])[["EFCPixels", "TotalLossPixels"]]
    .sum()
    .reset_index()
)
region_yearly["EFCR"] = region_yearly["EFCPixels"] / region_yearly["TotalLossPixels"]
region_yearly.replace([np.inf, -np.inf], np.nan, inplace=True)
region_yearly = region_yearly.dropna(subset=["EFCR", "TotalLossPixels"])

# === Summaries =================================================================================

records = []

# National summary
ny = national_yearly
mean_val, mean_lo, mean_hi = bootstrap_weighted_mean_ci(
    x=ny["EFCR"].values, w=ny["TotalLossPixels"].values, B=2000, alpha=0.05, random_state=42
)
sen = sen_slope_and_ci(ny["Year_From"].values, ny["EFCR"].values, alpha=0.05)
records.append({
    "Level": "National",
    "Region": "CONUS",
    "Years": f"{int(ny['Year_From'].min())}-{int(ny['Year_From'].max())}",
    "N_years": int(ny.shape[0]),
    "MeanEFCR": mean_val,
    "MeanEFCR_CI95_lo": mean_lo,
    "MeanEFCR_CI95_hi": mean_hi,
    "SenSlope": sen["slope"],
    "SenSlope_CI95_lo": sen["lo"],
    "SenSlope_CI95_hi": sen["hi"],
    "MK_pvalue": sen["p"],
})

# Regional summaries
for region in sorted(region_yearly["Region"].unique()):
    sub = region_yearly[region_yearly["Region"] == region].copy()
    if sub.empty:
        continue
    mean_val, mean_lo, mean_hi = bootstrap_weighted_mean_ci(
        x=sub["EFCR"].values, w=sub["TotalLossPixels"].values, B=2000, alpha=0.05, random_state=42
    )
    sen = sen_slope_and_ci(sub["Year_From"].values, sub["EFCR"].values, alpha=0.05)
    records.append({
        "Level": "Region",
        "Region": region,
        "Years": f"{int(sub['Year_From'].min())}-{int(sub['Year_From'].max())}",
        "N_years": int(sub.shape[0]),
        "MeanEFCR": mean_val,
        "MeanEFCR_CI95_lo": mean_lo,
        "MeanEFCR_CI95_hi": mean_hi,
        "SenSlope": sen["slope"],
        "SenSlope_CI95_lo": sen["lo"],
        "SenSlope_CI95_hi": sen["hi"],
        "MK_pvalue": sen["p"],
    })

national_and_regions_summary = pd.DataFrame.from_records(records)

# Optional: nice ordering of columns
cols = ["Level","Region","Years","N_years",
        "MeanEFCR","MeanEFCR_CI95_lo","MeanEFCR_CI95_hi",
        "SenSlope","SenSlope_CI95_lo","SenSlope_CI95_hi","MK_pvalue"]
national_and_regions_summary = national_and_regions_summary[cols].sort_values(["Level","Region"]).reset_index(drop=True)

print(national_and_regions_summary)